# Validación de QuEtool — Vina nativo como contraste

Este notebook corre **AutoDock Vina nativo** sobre los mismos once ligandos, el mismo
receptor y la misma caja que usa QuEtool, y deja los ΔG al lado de los de **Webina**
(el Vina compilado a WebAssembly que QuEtool lleva adentro).

Cumple dos funciones a la vez dentro de la validación:

1. **Tapa el único hueco metodológico del protocolo**: da una corrida nativa e
   independiente contra la cual contrastar los ΔG de QuEtool.
2. **Prueba que el port a WebAssembly no se desvía del binario oficial.** Webina *es*
   Vina 1.2.3 compilado a WASM; si los ΔG coinciden, QuEtool usa Vina de verdad.

Corre entero en Colab gratis, en CPU. No hace falta GPU: el `conf.txt` del TP es formato
Vina, así que el motor correcto es Vina 1.2.x en CPU, no AutoDock-GPU.

> Parte de la validación de QuEtool contra el TP de Química Medicinal (FFyB-UBA).
> La salida se guarda en `QuEtool - materiales TP reconstruidos/Validacion-Vina-nativo/`.


## 1. Montar el Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Encontrar la carpeta de materiales

Busca la carpeta por nombre dentro de *Mi unidad*. Si la tenés en otro lado, editá
`BASE` a mano con la ruta correcta.


In [ ]:
import os, glob

NOMBRE = 'QuEtool - materiales TP reconstruidos'
candidatos = glob.glob(f'/content/drive/MyDrive/**/{NOMBRE}', recursive=True) \
           + glob.glob(f'/content/drive/MyDrive/{NOMBRE}')
candidatos = sorted(set(candidatos))
assert candidatos, ('No encontré la carpeta. Editá BASE a mano con su ruta, '
                    'por ejemplo /content/drive/MyDrive/.../' + NOMBRE)
BASE = candidatos[0]
OUT  = os.path.join(BASE, 'Validacion-Vina-nativo')
os.makedirs(os.path.join(OUT, 'poses'), exist_ok=True)
print('Materiales:', BASE)
print('Receptor  :', os.path.exists(os.path.join(BASE, 'Receptor', '2B6A_receptor.pdbqt')))
print('Ligandos  :', len(glob.glob(os.path.join(BASE, 'Ligandos', '*.pdbqt'))), 'PDBQT')
print('Salida    :', OUT)


## 3. Bajar Vina nativo 1.2.3

La **misma versión** que Webina (1.2.3), para que la comparación sea exacta. Binario
estático oficial de los release de AutoDock-Vina en GitHub; no se instala nada.


In [ ]:
import subprocess, urllib.request, os
URL = 'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.3/vina_1.2.3_linux_x86_64'
VINA = '/content/vina'
if not os.path.exists(VINA):
    urllib.request.urlretrieve(URL, VINA)
    os.chmod(VINA, 0o755)
print(subprocess.run([VINA, '--version'], capture_output=True, text=True).stdout.strip())


## 4. Leer la caja del `conf.txt`

No se escriben los números a mano: se leen del mismo `conf.txt` que usa QuEtool, así
la caja es idéntica por construcción.


In [ ]:
conf = {}
with open(os.path.join(BASE, 'conf.txt')) as f:
    for ln in f:
        ln = ln.split('#')[0].strip()
        if '=' in ln:
            k, v = ln.split('=')
            conf[k.strip()] = float(v.strip())
CAJA = dict(center_x=conf['center_x'], center_y=conf['center_y'], center_z=conf['center_z'],
            size_x=conf['size_x'], size_y=conf['size_y'], size_z=conf['size_z'])
EXH  = int(conf.get('exhaustiveness', 10))
SEED = 42   # fija la semilla para que la corrida sea reproducible
print('Caja:', CAJA)
print('Exhaustiveness:', EXH, '| seed:', SEED)


## 5. Dockear los once ligandos

Receptor rígido, misma caja, misma exhaustiveness. Guarda las poses de cada uno en
`Validacion-Vina-nativo/poses/` y se queda con el ΔG de la mejor pose.


In [ ]:
import glob, os, re, time
REC = os.path.join(BASE, 'Receptor', '2B6A_receptor.pdbqt')
ligandos = sorted(glob.glob(os.path.join(BASE, 'Ligandos', '*.pdbqt')))
dG_nativo = {}
for lp in ligandos:
    nom = os.path.splitext(os.path.basename(lp))[0]
    out = os.path.join(OUT, 'poses', f'{nom}_vina.pdbqt')
    cmd = [VINA, '--receptor', REC, '--ligand', lp, '--out', out,
           '--exhaustiveness', str(EXH), '--seed', str(SEED), '--cpu', '2']
    for k, v in CAJA.items():
        cmd += ['--' + k, str(v)]
    t0 = time.time()
    subprocess.run(cmd, capture_output=True, text=True)
    # el ΔG de la mejor pose es el primer REMARK VINA RESULT del archivo de salida
    dg = None
    with open(out) as f:
        for ln in f:
            if 'VINA RESULT' in ln:
                dg = float(ln.split(':')[1].split()[0]); break
    dG_nativo[nom] = dg
    print(f'{nom:9s}  ΔG = {dg:7.2f} kcal/mol   ({time.time()-t0:4.0f} s)')


## 6. Tabla final: nativo vs Webina, con pIC50

Los ΔG de Webina se pegan en el diccionario `WEBINA` a medida que salen de la corrida
de QuEtool. T50 ya está (el control de redocking dio −12,501). Los demás se completan
con los valores de QuEtool y la tabla queda lista para el informe.


In [ ]:
import csv, math
# ΔG de QuEtool/Webina — se completa con la corrida de QuEtool (hoy sólo el control T50).
WEBINA = {'T50': -12.501}
# IC50 (nM) de la serie, de la guía del TP. T50 es control; EFZ-Br se predice.
IC50 = {'GFA':2.00,'EFZ':6.03,'TBO':14.13,'TNK':40.74,'NVP':89.13,'HBQ':177.83,
        'QNO47b':213.80,'U05':398.11,'ZZE':630.96,'QNO47c':912.01}
def pIC50(nM): return 9 - math.log10(nM)   # -log10(IC50 en M)

filas = []
for nom in sorted(dG_nativo):
    dn = dG_nativo[nom]; dw = WEBINA.get(nom)
    filas.append(dict(ligando=nom, dG_vina_nativo=dn, dG_webina=dw,
                      delta=(None if dw is None else round(dn-dw,3)),
                      IC50_nM=IC50.get(nom), pIC50=(round(pIC50(IC50[nom]),3) if nom in IC50 else None)))

csv_path = os.path.join(OUT, 'dG_vina_nativo_vs_webina.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['ligando','dG_vina_nativo','dG_webina','delta','IC50_nM','pIC50'])
    w.writeheader(); w.writerows(filas)
print('CSV guardado en:', csv_path)
print()
print(f"{'ligando':9s} {'nativo':>8s} {'webina':>8s} {'Δ':>7s} {'IC50(nM)':>9s} {'pIC50':>6s}")
for r in filas:
    f_ = lambda x, p='.2f': ('—' if x is None else format(x, p))
    print(f"{r['ligando']:9s} {f_(r['dG_vina_nativo']):>8s} {f_(r['dG_webina']):>8s} "
          f"{f_(r['delta'],'.3f'):>7s} {f_(r['IC50_nM'],'.2f'):>9s} {f_(r['pIC50'],'.3f'):>6s}")


## 7. Qué mirar

- **La columna `delta`** (nativo − Webina) debería ser chica para todos. Si lo es, el
  Vina de WebAssembly de QuEtool no se desvía del binario oficial: ésa es la prueba.
- **T50** es el control de redocking; su ΔG nativo tiene que caer cerca del −12,501 de
  Webina.
- Con la columna `pIC50` y los ΔG ya se puede ajustar `pIC50 = f(ΔG)` y contestar la
  pregunta del TP sobre si la relación es significativa.

Cuando termine, el CSV y las poses quedan en el Drive, en
`Validacion-Vina-nativo/`, listos para que el análisis los levante.
